In [1]:
import pandas as pd
import numpy as np
import pyodbc
import warnings

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 50)

def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
    return df

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    conn.cursor().execute("SELECT 1").fetchone()
print("ODBC connection OK")

ODBC connection OK


In [2]:
ragu_df = run_sql("""
    SELECT account_number, lob, book_date,
           gross_loss_ragu, apr, amt_financed, model_score
    FROM sandbox.gl_ragu_individual
""")

ragu_df['book_date'] = pd.to_datetime(ragu_df['book_date'])
print(f"Rows: {len(ragu_df):,}")
print(f"\nLOB distribution:")
print(ragu_df['lob'].value_counts())

print("\n--- gross_loss_ragu range by LOB ---")
print(ragu_df.groupby('lob')['gross_loss_ragu'].describe(
    percentiles=[0.25, 0.5, 0.75]
)[['min', '25%', '50%', '75%', 'max']].round(2))

Rows: 818,180

LOB distribution:
lob
KMX    429414
FRN    116750
ENT    101365
STG     88248
AN      49442
FLD     32961
Name: count, dtype: int64

--- gross_loss_ragu range by LOB ---
        min     25%     50%     75%     max
lob                                        
AN    97.70  131.93  138.69  145.00  192.50
ENT  102.04  128.69  133.88  139.12  192.12
FLD   99.96  133.67  140.04  145.75  204.50
FRN   92.00  134.10  140.71  146.50  209.50
KMX   78.00  133.08  139.81  146.72  235.26
STG   91.58  131.69  138.83  145.29  206.50


In [3]:
print("=" * 70)
print("APR DISTRIBUTION BY LOB")
print("=" * 70)
print(ragu_df.groupby('lob')['apr'].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)[['min', '10%', '25%', '50%', '75%', '90%', 'max']].round(4))

print(f"\n{'=' * 70}")
print("GROSS LOSS RAGU DISTRIBUTION BY LOB")
print("=" * 70)
print(ragu_df.groupby('lob')['gross_loss_ragu'].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)[['count', 'min', '10%', '25%', '50%', '75%', '90%', 'max']].round(2))

APR DISTRIBUTION BY LOB
        min     10%     25%     50%     75%     90%     max
lob                                                        
AN   0.1500  0.2060  0.2410  0.2597  0.2799  0.2800  0.2999
ENT  0.1425  0.1843  0.2131  0.2499  0.2499  0.2499  0.2799
FLD  0.1673  0.1818  0.2100  0.2468  0.2641  0.2799  0.2899
FRN  0.1500  0.2100  0.2410  0.2641  0.2799  0.2800  0.2999
KMX  0.0700  0.2000  0.2325  0.2641  0.2800  0.2800  0.2800
STG  0.1199  0.2060  0.2410  0.2700  0.2799  0.2800  0.2999

GROSS LOSS RAGU DISTRIBUTION BY LOB
        count     min     10%     25%     50%     75%     90%     max
lob                                                                  
AN    49442.0   97.70  126.00  131.93  138.69  145.00  151.50  192.50
ENT  101365.0  102.04  124.09  128.69  133.88  139.12  143.93  192.12
FLD   32961.0   99.96  126.96  133.67  140.04  145.75  151.52  204.50
FRN  116750.0   92.00  127.44  134.10  140.71  146.50  152.37  209.50
KMX  429414.0   78.00  127.00  133.08  

In [4]:
actuals_df = run_sql("""
    WITH all_accounts AS (
        SELECT account_number,
               sp_book_date_quarter_vintage,
               MAX(proceeds) AS proceeds
        FROM edwnpi.svc_master_monthend
        WHERE sp_book_date_quarter_vintage >= '2020 Q1'
        GROUP BY 1, 2
    ),
    chargeoffs AS (
        SELECT account_number,
               sp_book_date_quarter_vintage,
               MAX(gross_loss_amt) AS gross_loss_amt
        FROM edwnpi.svc_master_monthend
        WHERE sp_book_date_quarter_vintage >= '2020 Q1'
          AND account_life_cycle = 'CHARGEOFF'
        GROUP BY 1, 2
    )
    SELECT a.account_number,
           a.sp_book_date_quarter_vintage AS vintage,
           a.proceeds,
           COALESCE(co.gross_loss_amt, 0) AS gross_loss_amt,
           CASE WHEN co.account_number IS NOT NULL THEN 1 ELSE 0 END AS is_chargeoff
    FROM all_accounts a
    LEFT JOIN chargeoffs co
        ON a.account_number = co.account_number
       AND a.sp_book_date_quarter_vintage = co.sp_book_date_quarter_vintage
""")

print(f"Actuals rows: {len(actuals_df):,}")
print(f"Chargeoff rate: {actuals_df['is_chargeoff'].mean():.4f}")
print(f"Vintages: {actuals_df['vintage'].nunique()}")
print(f"Vintage range: {actuals_df['vintage'].min()} to {actuals_df['vintage'].max()}")

Actuals rows: 929,442
Chargeoff rate: 0.3512
Vintages: 27
Vintage range: 2020 Q1 to 225 Q4


In [5]:
merged = ragu_df.merge(actuals_df[['account_number', 'vintage', 'proceeds', 'gross_loss_amt']],
                       on='account_number', how='inner')

print(f"Pre-filter merged rows: {len(merged):,}")
print(f"Book date range in gl_ragu_individual: {merged['book_date'].min()} to {merged['book_date'].max()}")

# gl_ragu_individual only has 2024+ loans. Filter to loans with enough seasoning
# (booked by end of 2024 = at least 18 months of loss emergence by mid-2026)
MATURITY_CUTOFF = '2024-12-31'
merged = merged[merged['book_date'] <= MATURITY_CUTOFF].copy()

merged['gross_loss_pct'] = merged['gross_loss_amt'] / merged['proceeds']

# Quintiles per LOB
LOBS = sorted(merged['lob'].unique())
for lob in LOBS:
    mask = merged['lob'] == lob
    merged.loc[mask, 'ragu_quintile'] = pd.qcut(
        merged.loc[mask, 'gross_loss_ragu'], q=5, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4', 'Q5 (High)']
    )

# APR bands: 18% to 28% in 2% steps
apr_bins = [i / 100 for i in range(18, 30, 2)]
apr_labels = [f"{i}-{i+2}%" for i in range(18, 28, 2)]
merged['apr_band'] = pd.cut(merged['apr'], bins=apr_bins, labels=apr_labels, right=True)

print(f"\nAfter maturity filter (<= {MATURITY_CUTOFF}): {len(merged):,} rows")
print(f"Book date range: {merged['book_date'].min()} to {merged['book_date'].max()}")
print(f"\nLOB counts:")
print(merged['lob'].value_counts().sort_index())
print(f"\nAPR band distribution ({merged['apr_band'].notna().sum():,} in 18-28% range):")
print(merged['apr_band'].value_counts().sort_index())

Pre-filter merged rows: 805,967
Book date range in gl_ragu_individual: 2020-01-02 00:00:00 to 2026-06-30 00:00:00

After maturity filter (<= 2024-12-31): 577,952 rows
Book date range: 2020-01-02 00:00:00 to 2024-12-31 00:00:00

LOB counts:
lob
AN      33773
ENT     73239
FLD     19420
FRN     71640
KMX    318760
STG     61120
Name: count, dtype: int64

APR band distribution (544,445 in 18-28% range):
apr_band
18-20%     23440
20-22%     71964
22-24%     42577
24-26%    147821
26-28%    258643
Name: count, dtype: int64


In [6]:
print("=" * 70)
print("GROSS LOSS RAGU DISTRIBUTION BY LOB (merged dataset)")
print("=" * 70)

range_table = merged.groupby('lob').agg(
    count=('gross_loss_ragu', 'count'),
    min=('gross_loss_ragu', 'min'),
    p25=('gross_loss_ragu', lambda x: x.quantile(0.25)),
    median=('gross_loss_ragu', 'median'),
    p75=('gross_loss_ragu', lambda x: x.quantile(0.75)),
    max=('gross_loss_ragu', 'max'),
    mean_apr=('apr', 'mean'),
    actual_gl_rate=('gross_loss_pct', 'mean'),
).round(3)

display(range_table)

GROSS LOSS RAGU DISTRIBUTION BY LOB (merged dataset)


,count,min,p25,median,p75,max,mean_apr,actual_gl_rate
lob,,,,,,,,
AN,33773,97.698,130.246,136.582,143.334,192.500,0.254,0.352
ENT,73239,102.040,127.687,132.628,137.417,192.118,0.235,0.408
FLD,19420,99.962,131.091,138.051,144.500,204.500,0.236,0.319
FRN,71640,92.000,131.438,137.959,144.367,209.500,0.255,0.441
KMX,318760,78.000,132.011,139.213,146.562,235.262,0.250,0.379
STG,61120,91.582,129.687,136.761,143.752,206.500,0.256,0.385


In [7]:
def build_pivot_by_lob(df, lob):
    """Build actual GL rate pivot for a single LOB with quintile RAGU ranges."""
    lob_df = df[(df['lob'] == lob) & df['apr_band'].notna()].copy()
    if len(lob_df) == 0:
        print(f"\n{lob}: no data in APR range")
        return None, None

    # Quintile range summary
    q_ranges = lob_df.groupby('ragu_quintile', observed=True)['gross_loss_ragu'].agg(['min', 'max', 'count'])
    q_ranges.columns = ['ragu_min', 'ragu_max', 'n']
    q_ranges['range_label'] = q_ranges.apply(lambda r: f"[{r.ragu_min:.1f} - {r.ragu_max:.1f}]", axis=1)

    grouped = lob_df.groupby(['ragu_quintile', 'apr_band'], observed=True).agg(
        total_gl=('gross_loss_amt', 'sum'),
        total_proceeds=('proceeds', 'sum'),
        loan_count=('account_number', 'count')
    ).reset_index()
    grouped['gl_rate'] = grouped['total_gl'] / grouped['total_proceeds']

    pivot_rate = grouped.pivot(index='ragu_quintile', columns='apr_band', values='gl_rate')
    pivot_count = grouped.pivot(index='ragu_quintile', columns='apr_band', values='loan_count')

    # Attach RAGU range as first column
    display_rate = pivot_rate.copy() * 100
    display_rate.insert(0, 'RAGU Range', q_ranges['range_label'])
    display_rate.insert(1, 'N', q_ranges['n'])

    print(f"\n{'=' * 80}")
    print(f"ACTUAL GROSS LOSS RATE (%) -- {lob}")
    print(f"{'=' * 80}")
    display(display_rate.round(3))

    return pivot_rate, pivot_count

lob_results = {}
for lob in LOBS:
    rate, count = build_pivot_by_lob(merged, lob)
    if rate is not None:
        lob_results[lob] = {'rate': rate, 'count': count}

# Combined Non-KMX table
nonkmx_df = merged[(merged['lob'] != 'KMX') & merged['apr_band'].notna()].copy()
nonkmx_df['ragu_quintile'] = pd.qcut(
    nonkmx_df['gross_loss_ragu'], q=5, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4', 'Q5 (High)']
)

q_ranges = nonkmx_df.groupby('ragu_quintile', observed=True)['gross_loss_ragu'].agg(['min', 'max', 'count'])
q_ranges.columns = ['ragu_min', 'ragu_max', 'n']
q_ranges['range_label'] = q_ranges.apply(lambda r: f"[{r.ragu_min:.1f} - {r.ragu_max:.1f}]", axis=1)

grouped = nonkmx_df.groupby(['ragu_quintile', 'apr_band'], observed=True).agg(
    total_gl=('gross_loss_amt', 'sum'),
    total_proceeds=('proceeds', 'sum'),
    loan_count=('account_number', 'count')
).reset_index()
grouped['gl_rate'] = grouped['total_gl'] / grouped['total_proceeds']

nonkmx_pivot_rate = grouped.pivot(index='ragu_quintile', columns='apr_band', values='gl_rate')

display_rate = nonkmx_pivot_rate.copy() * 100
display_rate.insert(0, 'RAGU Range', q_ranges['range_label'])
display_rate.insert(1, 'N', q_ranges['n'])

print(f"\n{'=' * 80}")
print(f"ACTUAL GROSS LOSS RATE (%) -- NON-KMX COMBINED")
print(f"{'=' * 80}")
display(display_rate.round(3))

lob_results['Non-KMX'] = {'rate': nonkmx_pivot_rate}


ACTUAL GROSS LOSS RATE (%) -- AN


apr_band,RAGU Range,N,18-20%,20-22%,22-24%,24-26%,26-28%
ragu_quintile,,,,,,,
Q1 (Low),[97.7 - 128.4],6729,33.890,45.011,52.834,53.402,54.183
Q2,[128.4 - 134.2],6575,31.903,37.226,53.316,44.384,45.691
Q3,[134.3 - 139.1],6483,33.017,33.402,42.426,37.993,37.989
Q4,[139.1 - 145.0],6376,22.903,29.727,29.273,30.912,30.331
Q5 (High),[145.0 - 192.5],6260,14.172,18.374,19.381,17.631,18.132



ACTUAL GROSS LOSS RATE (%) -- ENT


apr_band,RAGU Range,N,18-20%,20-22%,22-24%,24-26%,26-28%
ragu_quintile,,,,,,,
Q1 (Low),[102.0 - 126.6],14169,49.998,53.308,58.629,59.082,61.562
Q2,[126.6 - 130.7],13938,46.909,46.208,48.106,50.377,58.151
Q3,[130.7 - 134.4],13519,40.931,38.500,41.424,42.268,53.347
Q4,[134.4 - 138.7],13087,35.010,30.803,31.406,34.360,44.055
Q5 (High),[138.7 - 187.7],11664,23.384,20.822,24.542,25.167,29.073



ACTUAL GROSS LOSS RATE (%) -- FLD


...

apr_band,RAGU Range,N,18-20%,20-22%,22-24%,24-26%,26-28%
ragu_quintile,,,,,,,
Q1 (Low),[78.0 - 130.3],63289,45.851,48.735,51.770,54.135,55.030
Q2,[130.3 - 136.5],62731,39.827,40.501,43.556,44.995,45.909
Q3,[136.5 - 142.0],61562,33.009,32.153,36.562,37.672,38.885
Q4,[142.0 - 148.5],58730,27.020,27.218,29.636,32.905,33.017
Q5 (High),[148.5 - 235.3],54499,18.667,20.285,20.454,24.544,24.347



ACTUAL GROSS LOSS RATE (%) -- STG


apr_band,RAGU Range,N,18-20%,20-22%,22-24%,24-26%,26-28%
ragu_quintile,,,,,,,
Q1 (Low),[91.6 - 127.7],12100,56.600,48.941,68.019,59.160,61.737
Q2,[127.7 - 134.3],12098,50.591,39.892,44.447,46.034,51.304
Q3,[134.3 - 139.4],11337,33.758,33.840,41.583,36.433,41.844
Q4,[139.4 - 145.6],11332,24.925,26.520,34.694,29.882,34.082
Q5 (High),[145.6 - 206.5],10966,17.158,17.179,23.301,18.099,21.294



ACTUAL GROSS LOSS RATE (%) -- NON-KMX COMBINED


apr_band,RAGU Range,N,18-20%,20-22%,22-24%,24-26%,26-28%
ragu_quintile,,,,,,,
Q1 (Low),[91.6 - 127.6],48735,49.138,52.279,58.517,58.874,61.470
Q2,[127.6 - 133.0],48931,45.818,43.303,48.570,47.514,52.792
Q3,[133.0 - 137.8],48576,35.352,35.703,42.731,40.224,45.780
Q4,[137.8 - 143.6],48668,29.622,29.605,35.847,31.825,37.874
Q5 (High),[143.6 - 209.5],48724,21.245,20.315,27.812,22.305,24.595


In [8]:
from scipy.stats import linregress

apr_midpoints = {f"{i}-{i+2}%": (i + 1) / 100 for i in range(18, 28, 2)}

EXPECTED_MULT = {'KMX': 0.7 / 0.65, 'AN': 0.7, 'ENT': 0.7, 'FLD': 0.7, 'FRN': 0.7, 'STG': 0.7}

def implied_multiplier(pivot_rate, lob):
    """Regress actual GL rate on APR midpoint within each quintile."""
    expected_mult = EXPECTED_MULT.get(lob, 0.7)
    print(f"\n{'=' * 80}")
    print(f"IMPLIED APR MULTIPLIER -- {lob} (expected: {expected_mult:.3f})")
    print(f"{'=' * 80}")
    print(f"{'Quintile':<12} {'Slope (per 1pp)':<18} {'Implied Mult':<15} {'vs Expected':<15} {'R-sq':<8}")
    print("-" * 68)

    for quintile in pivot_rate.index:
        row = pivot_rate.loc[quintile]
        valid = row.dropna()
        if len(valid) < 3:
            print(f"{quintile:<12} {'insufficient data':<50}")
            continue

        x = np.array([apr_midpoints[col] for col in valid.index])
        y = valid.values

        slope, intercept, r_value, p_value, std_err = linregress(x, y)

        # slope is d(GL_rate)/d(APR); negative slope = higher APR means lower GL
        # Implied mult = -slope / 0.01 (per-1pp, score-point scale)
        implied = -slope / 0.01

        ratio = implied / expected_mult if expected_mult != 0 else np.nan
        direction = 'over' if ratio > 1 else 'under'

        print(f"{quintile:<12} {slope:>+.6f}{'':>8} {implied:>+.4f}{'':>7} "
              f"{ratio:.2f}x ({direction})   {r_value**2:.3f}")

for lob in LOBS:
    if lob in lob_results:
        implied_multiplier(lob_results[lob]['rate'], lob)

if 'Non-KMX' in lob_results:
    implied_multiplier(lob_results['Non-KMX']['rate'], 'Non-KMX')


IMPLIED APR MULTIPLIER -- AN (expected: 0.700)
Quintile     Slope (per 1pp)    Implied Mult    vs Expected     R-sq    
--------------------------------------------------------------------
Q1 (Low)     +2.448866         -244.8866        -349.84x (under)   0.803
Q2           +1.736676         -173.6676        -248.10x (under)   0.445
Q3           +0.726778         -72.6778        -103.83x (under)   0.351
Q4           +0.802109         -80.2109        -114.59x (under)   0.605
Q5 (High)    +0.358867         -35.8867        -51.27x (under)   0.326

IMPLIED APR MULTIPLIER -- ENT (expected: 0.700)
Quintile     Slope (per 1pp)    Implied Mult    vs Expected     R-sq    
--------------------------------------------------------------------
Q1 (Low)     +1.445137         -144.5137        -206.45x (under)   0.936
Q2           +1.332681         -133.2681        -190.38x (under)   0.755
Q3           +1.429973         -142.9973        -204.28x (under)   0.610
Q4           +1.082378         -108.237